In [10]:
import json
import pandas as pd
# Load the test data from test.json
with open("test.json", "r") as f:
    test_data = json.load(f)
# Convert the test data to a DataFrame
test_df = pd.DataFrame(test_data)
# Ensure the DataFrame has the required columns
print(test_df.head())

                                             content  \
0  Vijay Shinde\nasst. sales manager in M - S PLA...   
1  Jaykumar Shah\nSales Manager\n\nMumbai, Mahara...   
2  Raktim Podder\n6+ Exp in banking operations an...   
3  Bhaskar Gupta\nBusiness Management professiona...   
4  Rohit Bijlani\nJAVA Developer/Senior Systems E...   

                                          annotation extras  \
0  [{'label': ['Email Address'], 'points': [{'sta...   None   
1  [{'label': ['Skills'], 'points': [{'start': 38...   None   
2  [{'label': ['Skills'], 'points': [{'start': 88...   None   
3  [{'label': ['Companies worked at'], 'points': ...   None   
4  [{'label': ['Graduation Year'], 'points': [{'s...   None   

                                            metadata  
0  {'first_done_at': 1534860188000, 'last_updated...  
1  {'first_done_at': 1533400611000, 'last_updated...  
2  {'first_done_at': 1528734410000, 'last_updated...  
3  {'first_done_at': 1536754903000, 'last_updated...  
4  {'firs

In [11]:
for i, row in test_df.iterrows():
    print(f"Resume {i}:")
    for item in row['annotation']:
        print(f"  Label: {item['label']}, Text: {[p['text'] for p in item['points']]}")
    print()

Resume 0:
  Label: ['Email Address'], Text: ['indeed.com/r/Vijay-Shinde/3981b85e9130be2b']
  Label: ['Companies worked at'], Text: ['S PLANET TECHNOLIES KHOPOLI']
  Label: ['Designation'], Text: ['asst. sales manager']
  Label: ['Email Address'], Text: ['indeed.com/r/Vijay-Shinde/3981b85e9130be2b']
  Label: ['Location'], Text: ['Panvel']
  Label: ['Companies worked at'], Text: ['S PLANET TECHNOLIES KHOPOLI']
  Label: ['Designation'], Text: ['asst. sales manager']
  Label: ['Name'], Text: ['Vijay Shinde']

Resume 1:
  Label: ['Skills'], Text: ['Well Versed with Internet and E-mail']
  Label: ['Skills'], Text: ['Exposure to computerized working environment.']
  Label: ['Skills'], Text: ['PowerPoint']
  Label: ['Skills'], Text: [' Excel']
  Label: ['Skills'], Text: [' MS - Office']
  Label: ['Skills'], Text: ['Working knowledge of windows']
  Label: ['Skills'], Text: ['POWERPOINT']
  Label: ['Skills'], Text: ['EXCEL']
  Label: ['Skills'], Text: ['CORRESPONDENCE']
  Label: ['Skills'], Text

In [12]:
import torch
from transformers import BertTokenizer, BertForTokenClassification, BertConfig
# Define the tag2idx mapping
tag2idx = {
    'I-COMPANY': 0,  'U-COMPANY': 1,   'I-EMAIL': 2,     'I-GRADYEAR': 3,
    'L-EMAIL': 4,    '[SEP]': 5,       'B-CLG': 6,       'U-SKILLS': 7,
    'X': 8,          'L-NAME': 9,      'B-DEG': 10,      'U-EMAIL': 11,
    'L-CLG': 12,     'L-GRADYEAR': 13, 'L-DESIG': 14,    'U-DEG': 15,
    'B-SKILLS': 16,  'L-LOC': 17,      'I-SKILLS': 18,   'U-GRADYEAR': 19,
    'B-NAME': 20,    'I-YOE': 21,      'B-YOE': 22,      'L-YOE': 23,
    'I-DEG': 24,     'B-COMPANY': 25,  'I-NAME': 26,     'I-DESIG': 27,
    'U-LOC': 28,     'B-GRADYEAR': 29, 'O': 30,          'U-YOE': 31,
    'I-CLG': 32,     'L-SKILLS': 33,   'B-LOC': 34,      'L-DEG': 35,
    'U-CLG': 36,     'U-DESIG': 37,    'B-EMAIL': 38,    'B-DESIG': 39,
    '[CLS]': 40,     'I-LOC': 41,      'L-COMPANY': 42
}

In [13]:
# Load tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-cased', do_lower_case=False)

In [14]:
# Inverse mapping
idx2tag = {v: k for k, v in tag2idx.items()}

In [15]:
# Sample resume content
text = test_df.iloc[0]['content']

In [16]:
# Tokenize input
inputs = tokenizer.encode_plus(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=512,
    is_split_into_words=False,
    return_attention_mask=True
)

In [26]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
inputs = {k: v.to(device) for k, v in inputs.items()}

In [27]:
# Your tag2idx must be already defined above this
config = BertConfig(
    vocab_size=28996,
    num_labels=len(tag2idx),
    hidden_size=768,
    num_attention_heads=12,
    num_hidden_layers=12
)
model = BertForTokenClassification(config)

In [28]:
# Load model weights (adjust path as needed)
checkpoint = torch.load("fifth_epoch.pt", map_location="cpu")
model.load_state_dict(checkpoint, strict=False)

_IncompatibleKeys(missing_keys=[], unexpected_keys=['bert.pooler.dense.weight', 'bert.pooler.dense.bias'])

In [29]:
model.eval()

BertForTokenClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12

In [30]:
# Predict
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits
    predicted_ids = torch.argmax(logits, dim=2).squeeze().tolist()

RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cpu and cuda:0! (when checking argument for argument index in method wrapper_CUDA__index_select)

In [28]:
# Convert tokens and predictions to readable format
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"].squeeze())
predicted_tags = [idx2tag[pred_id] for pred_id in predicted_ids]

In [1]:
# Display token-tag pairs (skip special tokens optionally)
for token, tag in zip(tokens, predicted_tags):
    if tag not in ['[CLS]', '[SEP]', 'X']:  # optional filter
        print(f"{token:15} --> {tag}")

NameError: name 'tokens' is not defined

In [12]:

# Define the configuration (use the same configuration as in training)
config = BertConfig(
    vocab_size=28996,       # Match the vocab size used during training
    num_labels=len(tag2idx),  # Number of classes in your classification task
    hidden_size=768,        # Standard for BERT-base
    num_attention_heads=12,
    num_hidden_layers=12
)
# Initialize the model with the configuration
model = BertForTokenClassification(config)
# Load the state dictionary with strict=False to ignore unexpected keys
checkpoint = torch.load("fifth_epoch.pt", map_location="cpu")
model.load_state_dict(checkpoint, strict=False)
# Send model to CPU for debugging
device = torch.device("cpu")
model.to(device)
model.eval()

BertForTokenClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12

In [13]:
from torch.utils.data import Dataset, DataLoader
# Initialize tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
# Custom Dataset
class ResumeDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len=512):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0)
        }
# Load test texts
texts = test_df["content"].tolist()
# Create Dataset and DataLoader
test_dataset = ResumeDataset(texts, tokenizer)
test_loader = DataLoader(test_dataset, batch_size=16)

In [14]:
from tqdm import tqdm
# Prediction loop
all_preds = []
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        # Debugging statements
        print("Input IDs shape:", input_ids.shape)
        print("Attention Mask shape:", attention_mask.shape)
        print("Input IDs:", input_ids)
        print("Attention Mask:", attention_mask)
        # Check for NaN or Inf values
        if torch.isnan(input_ids).any() or torch.isinf(input_ids).any():
            raise ValueError("Input IDs contain NaN or Inf values.")
        if torch.isnan(attention_mask).any() or torch.isinf(attention_mask).any():
            raise ValueError("Attention Mask contains NaN or Inf values.")
        try:
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            preds = torch.argmax(logits, dim=2).cpu().numpy()
            all_preds.extend(preds)
        except Exception as e:
            print(f"Error processing batch: {e}")
            continue
# Done 🎉
print("✅ Predictions complete!")
print(f"Total predictions: {len(all_preds)}")
print(f"First 10 predictions: {all_preds[:10]}")

Testing:  14%|██████████▋                                                                | 1/7 [00:00<00:02,  2.97it/s]

Input IDs shape: torch.Size([16, 512])
Attention Mask shape: torch.Size([16, 512])
Input IDs: tensor([[  101, 17027, 12277,  ...,     0,     0,     0],
        [  101,  6108, 18494,  ...,  2372,  1012,   102],
        [  101, 10958, 22462,  ...,  1012,  2537,   102],
        ...,
        [  101, 12388,  2078,  ...,  3247,  9338,   102],
        [  101, 22827, 10932,  ...,     0,     0,     0],
        [  101,  7663, 14220,  ...,  1010, 10594,   102]])
Attention Mask: tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        ...,
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 1, 1, 1]])
Error processing batch: index out of range in self


Testing:  29%|█████████████████████▍                                                     | 2/7 [00:00<00:01,  3.42it/s]

Input IDs shape: torch.Size([16, 512])
Attention Mask shape: torch.Size([16, 512])
Input IDs: tensor([[  101, 28919,  9080,  ...,     0,     0,     0],
        [  101,  5034, 20668,  ...,  2000,  5504,   102],
        [  101, 21871,  8322,  ...,  2144,  2268,   102],
        ...,
        [  101,  2019,  4014,  ..., 10200,  1006,   102],
        [  101,  4086,  2100,  ...,     0,     0,     0],
        [  101, 14880,  9126,  ...,     0,     0,     0]])
Attention Mask: tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        ...,
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]])
Error processing batch: index out of range in self


Testing:  43%|████████████████████████████████▏                                          | 3/7 [00:00<00:01,  3.48it/s]

...

Input IDs shape: torch.Size([16, 512])
Attention Mask shape: torch.Size([16, 512])
Input IDs: tensor([[  101,  5506, 24572,  ...,     0,     0,     0],
        [  101, 10958, 21886,  ...,     0,     0,     0],
        [  101, 10556, 15265,  ...,  2044,  4341,   102],
        ...,
        [  101,  2168,  2121,  ...,  2254,  2297,   102],
        [  101,  1047,  9825,  ...,     0,     0,     0],
        [  101,  6583,  2615,  ...,     0,     0,     0]])
Attention Mask: tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 1, 1, 1],
        ...,
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]])
Error processing batch: index out of range in self


Testing:  71%|█████████████████████████████████████████████████████▌                     | 5/7 [00:01<00:00,  3.49it/s]

Input IDs shape: torch.Size([16, 512])
Attention Mask shape: torch.Size([16, 512])
Input IDs: tensor([[  101, 23233, 21564,  ...,  2037,  2836,   102],
        [  101,  7997,  8320,  ...,     0,     0,     0],
        [  101, 28916, 28919,  ...,  1022,  1010,   102],
        ...,
        [  101,  2158, 16084,  ...,  1038,  2475,   102],
        [  101,  2474, 15985,  ...,  2004,  2092,   102],
        [  101, 21358, 28029,  ...,     0,     0,     0]])
Attention Mask: tensor([[1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 1, 1, 1],
        ...,
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 0, 0, 0]])
Error processing batch: index out of range in self


Testing: 100%|███████████████████████████████████████████████████████████████████████████| 7/7 [00:01<00:00,  3.64it/s]

Input IDs shape: torch.Size([16, 512])
Attention Mask shape: torch.Size([16, 512])
Input IDs: tensor([[  101,  2474, 27488,  ...,  7663, 23451,   102],
        [  101,  5506,  3270,  ...,     0,     0,     0],
        [  101, 14021, 15202,  ...,  1006,  2625,   102],
        ...,
        [  101,  8038, 15222,  ...,  2968,  1010,   102],
        [  101, 26927, 23147,  ...,  2640,  7060,   102],
        [  101,  5003, 15689,  ...,  6337,  2094,   102]])
Attention Mask: tensor([[1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 1, 1, 1],
        ...,
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1]])
Error processing batch: index out of range in self
Input IDs shape: torch.Size([10, 512])
Attention Mask shape: torch.Size([10, 512])
Input IDs: tensor([[  101, 14022, 11937,  ...,     0,     0,     0],
        [  101,  2089, 14900,  ...,     0,     0,     0],
        [  101,  6683, 10105,  ...,  4002,  2629,

In [17]:
# Assuming 'label' is the column containing ground truth labels in test_df
ground_truth_labels = test_df["label"].tolist()
# Flatten the ground truth labels and predictions
flat_ground_truth = [item for sublist in ground_truth_labels for item in sublist]
flat_preds = [item for sublist in all_preds for item in sublist]
# Calculate accuracy
correct_predictions = sum(p == l for p, l in zip(flat_preds, flat_ground_truth))
accuracy = correct_predictions / len(flat_ground_truth)
print(f"Accuracy: {accuracy:.4f}")

KeyError: 'label'